# 📚 Agente de Estudos RAG — Tutorial Completo

**Projeto prático: Construindo um agente de estudos com RAG, FastAPI, Qdrant e Deploy em produção.**

Neste notebook, você vai entender **cada linha de código** do projeto, os conceitos de Engenharia de Dados por trás, e como tudo se conecta em uma aplicação real.

---

## 🗺️ Arquitetura do Projeto

```
┌─────────────┐     ┌──────────────┐     ┌───────────────┐     ┌──────────────┐
│  PDF Upload  │────▶│  Chunking    │────▶│  Embeddings   │────▶│   Qdrant     │
│  (Frontend)  │     │  (Python)    │     │  (OpenAI API) │     │   Cloud      │
└─────────────┘     └──────────────┘     └───────────────┘     └──────┬───────┘
                                                                       │
┌─────────────┐     ┌──────────────┐     ┌───────────────┐            │
│  Resposta    │◀────│  LLM (GPT)   │◀────│  Prompt +     │◀───────────┘
│  (Frontend)  │     │  Generation  │     │  Contexto     │  (Retrieval)
└─────────────┘     └──────────────┘     └───────────────┘
```

**Stack:**
- **Backend:** FastAPI (Python)
- **Embeddings:** OpenAI `text-embedding-3-small`
- **Vector Store:** Qdrant Cloud
- **LLM:** GPT-4o-mini
- **Deploy:** Render (Docker)
- **Frontend:** Lovable

---
## 📦 Parte 1 — Instalação e Setup

Antes de tudo, instalamos as dependências do projeto.

In [ ]:
# Instala todas as dependências do projeto
# O '!' no início executa comandos do terminal dentro do notebook
!pip install fastapi uvicorn python-dotenv openai qdrant-client pypdf python-multipart pydantic -q

In [ ]:
# Configura as variáveis de ambiente
# ⚠️ Em produção, NUNCA coloque API keys no código. Use .env ou secrets manager.
# Aqui no notebook é só pra fins didáticos.

import os

os.environ["OPENAI_API_KEY"] = "sk-..."      # 👈 cole sua key aqui
os.environ["QDRANT_URL"] = "https://..."      # 👈 URL do seu cluster Qdrant
os.environ["QDRANT_API_KEY"] = "..."          # 👈 API key do Qdrant
os.environ["COLLECTION_NAME"] = "study_agent"

print("✅ Variáveis de ambiente configuradas.")

---
## 🔤 Parte 2 — Embeddings: Transformando Texto em Vetores

### O que é um embedding?

Um **embedding** é uma representação numérica (vetor) de um texto. Textos com significados parecidos ficam "próximos" no espaço vetorial.

Exemplo:
- `"gato"` e `"felino"` → vetores próximos (alta similaridade)
- `"gato"` e `"SQL"` → vetores distantes (baixa similaridade)

### Por que `text-embedding-3-small`?

| Modelo | Dimensões | Custo/1M tokens | Uso ideal |
|--------|-----------|-----------------|----------|
| text-embedding-3-small | 1536 | ~$0.02 | RAG, busca semântica |
| text-embedding-3-large | 3072 | ~$0.13 | Alta precisão |

Para nosso caso, o `small` é mais que suficiente e cabe no free tier do Qdrant.

In [ ]:
from openai import OpenAI

client = OpenAI()

# Gerando embeddings para textos de exemplo
textos = [
    "Um pipeline de ETL extrai dados de fontes diversas",
    "ETL significa Extract, Transform, Load",
    "O bolo de chocolate leva farinha e cacau",
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=textos,
)

# Cada texto virou um vetor de 1536 números
for i, emb in enumerate(response.data):
    print(f"Texto {i+1}: {len(emb.embedding)} dimensões | Primeiros 5 valores: {emb.embedding[:5]}")

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    """Calcula a similaridade de cosseno entre dois vetores."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

embs = [item.embedding for item in response.data]

# Comparando similaridades
print(f"ETL vs ETL (sinônimos):  {cosine_similarity(embs[0], embs[1]):.4f}")  # alto!
print(f"ETL vs Bolo (sem relação): {cosine_similarity(embs[0], embs[2]):.4f}")  # baixo!

# 👆 Isso é a base de TODA busca semântica.
# O Qdrant faz exatamente isso, mas otimizado pra milhões de vetores.

---
## 🗄️ Parte 3 — Qdrant: Banco de Dados Vetorial

### Por que um banco vetorial?

Bancos relacionais (PostgreSQL, MySQL) são ótimos pra busca exata:
```sql
SELECT * FROM docs WHERE titulo = 'ETL'
```

Mas e se o usuário perguntar **"como dados são movidos entre sistemas"**?
Isso é semanticamente igual a ETL, mas nenhum `WHERE` encontraria.

O banco vetorial resolve isso: ele busca por **similaridade de significado**, não por match exato.

### Por que Qdrant?

- Free tier generoso (1GB, suficiente pra milhares de documentos)
- API REST + client Python oficial
- Usa HNSW (Hierarchical Navigable Small World) pra busca rápida
- Suporta filtros (metadata filtering) — essencial em produção

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Conectando ao Qdrant Cloud
qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)

COLLECTION = "study_agent"

# Criando a collection (equivalente a CREATE TABLE)
# Só precisa fazer uma vez — se já existir, pula
collections = [c.name for c in qdrant.get_collections().collections]

if COLLECTION not in collections:
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(
            size=1536,              # deve bater com as dimensões do embedding model
            distance=Distance.COSINE # métrica de similaridade
        ),
    )
    print(f"✅ Collection '{COLLECTION}' criada!")
else:
    print(f"ℹ️  Collection '{COLLECTION}' já existe.")

# Verificando
info = qdrant.get_collection(COLLECTION)
print(f"📊 Pontos armazenados: {info.points_count}")

---
## ✂️ Parte 4 — Chunking: Fatiando Documentos

### Por que não jogar o documento inteiro pro embedding?

1. **Limite de tokens**: modelos de embedding têm limite (8191 tokens no small)
2. **Precisão na busca**: chunks menores são mais específicos → retrieval mais relevante
3. **Contexto do LLM**: o prompt aumentado precisa caber no context window

### Estratégia: janela deslizante com overlap

```
Texto: [A B C D E F G H I J K L M N O P Q R S T]

Chunk 1: [A B C D E F G H I J]       (posição 0-9)
Chunk 2:         [I J K L M N O P Q R] (posição 8-17) ← overlap de 2 palavras
Chunk 3:                 [Q R S T]     (posição 16-19)
```

O **overlap** garante que informações na fronteira entre chunks não sejam perdidas.

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    """
    Fatia texto em chunks com overlap.
    
    Args:
        text: texto bruto
        chunk_size: tamanho máximo de cada chunk em palavras
        overlap: quantas palavras se repetem entre chunks consecutivos
    
    Returns:
        lista de dicts com 'text' e 'chunk_index'
    """
    words = text.split()
    chunks = []
    start = 0
    idx = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)

        if chunk_text.strip():
            chunks.append({"text": chunk_text, "chunk_index": idx})
            idx += 1

        start += chunk_size - overlap  # a "janela desliza"

    return chunks

# Teste com um texto fictício
texto_exemplo = "Um data lake é um repositório centralizado " * 200  # texto longo

chunks = chunk_text(texto_exemplo, chunk_size=100, overlap=20)
print(f"📄 Texto original: {len(texto_exemplo.split())} palavras")
print(f"✂️  Chunks gerados: {len(chunks)}")
print(f"📏 Tamanho do chunk 0: {len(chunks[0]['text'].split())} palavras")

---
## 📥 Parte 5 — Ingestão Completa: PDF → Qdrant

Agora vamos juntar tudo: ler um PDF, fatiar, gerar embeddings e gravar no Qdrant.

Este é o pipeline de **ETL** do nosso RAG:
- **E**xtract: `pypdf` lê o PDF
- **T**ransform: chunking + embedding
- **L**oad: upsert no Qdrant

In [ ]:
import uuid
import hashlib
from pypdf import PdfReader
import io

def extract_text_from_pdf(file_path: str) -> str:
    """Extrai texto de todas as páginas de um PDF."""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text

def generate_embeddings(texts: list[str]) -> list[list[float]]:
    """Gera embeddings em batch."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
    )
    return [item.embedding for item in response.data]

def deterministic_id(text: str) -> str:
    """Gera ID determinístico — garante idempotência (reingerir não duplica)."""
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, hashlib.md5(text.encode()).hexdigest()))

# ──────────────────────────────────────────────────────────
# 📌 EXERCÍCIO: Faça upload de um PDF aqui
#    No Google Colab, use files.upload()
#    Local, coloque o caminho do arquivo
# ──────────────────────────────────────────────────────────

# from google.colab import files
# uploaded = files.upload()  # descomente no Colab
# pdf_path = list(uploaded.keys())[0]

pdf_path = "seu_documento.pdf"  # 👈 troque pelo seu arquivo

print("Se você não tem um PDF, crie um de teste com o próximo bloco. ⬇️")

In [ ]:
# 🧪 Criando um PDF de teste com conteúdo sobre Engenharia de Dados
# (caso você não tenha um PDF à mão)

!pip install fpdf2 -q
from fpdf import FPDF

conteudo_estudo = """
Engenharia de Dados: Conceitos Fundamentais

1. Pipeline de ETL
ETL significa Extract, Transform, Load. É o processo de extrair dados de fontes diversas,
transformá-los (limpar, normalizar, enriquecer) e carregá-los em um destino como um data
warehouse ou data lake. Ferramentas populares incluem Apache Airflow, dbt e Apache Spark.

2. Data Lake vs Data Warehouse
Um Data Lake armazena dados brutos em qualquer formato (estruturado, semi-estruturado,
não-estruturado). É ideal para exploração e machine learning. Um Data Warehouse armazena
dados estruturados e otimizados para consultas analíticas (OLAP). Exemplos: Snowflake,
BigQuery, Redshift.

3. Apache Spark
Framework de processamento distribuído para big data. Processa dados em memória (muito mais
rápido que MapReduce). Suporta batch e streaming. PySpark é a interface Python do Spark.

4. Orquestração com Apache Airflow
Airflow permite definir, agendar e monitorar workflows (DAGs). Cada task em um DAG é uma
unidade de trabalho. O scheduler decide quando executar. Muito usado em pipelines de dados.

5. Streaming de Dados
Processamento de dados em tempo real usando ferramentas como Apache Kafka, Apache Flink ou
Spark Structured Streaming. Essencial para dashboards em tempo real, detecção de fraude e
sistemas de recomendação.

6. Modelagem Dimensional
Técnica de organizar dados em um data warehouse usando fatos (métricas) e dimensões
(contextos). O modelo estrela (star schema) tem uma tabela fato central conectada a
tabelas dimensão. Criado por Ralph Kimball.

7. Data Quality
Garantir que os dados sejam acurados, completos, consistentes e atualizados. Ferramentas
como Great Expectations e dbt tests automatizam validações. Em pipelines de produção,
data quality é tão importante quanto o pipeline em si.

8. Vector Databases e RAG
Bancos vetoriais como Qdrant, Pinecone e Weaviate armazenam embeddings — representações
numéricas de textos. RAG (Retrieval-Augmented Generation) combina busca vetorial com LLMs
para gerar respostas fundamentadas em documentos. É a ponte entre engenharia de dados e IA.

9. APIs e Microserviços
APIs REST permitem que sistemas se comuniquem. FastAPI é um framework Python moderno para
criar APIs com documentação automática, validação de tipos e alta performance. Em engenharia
de dados, APIs são usadas para expor modelos de ML, servir dados processados e integrar sistemas.

10. Deploy e MLOps
Colocar modelos e pipelines em produção envolve containerização (Docker), CI/CD, monitoramento
e versionamento. Plataformas como Render, AWS, GCP e Azure facilitam o deploy. MLOps é a
disciplina que une DevOps com Machine Learning.
"""

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=11)
for line in conteudo_estudo.strip().split("\n"):
    pdf.cell(0, 8, line, new_x="LMARGIN", new_y="NEXT")
pdf.output("engenharia_dados_estudo.pdf")

pdf_path = "engenharia_dados_estudo.pdf"
print(f"✅ PDF de teste criado: {pdf_path}")

In [ ]:
# ── Pipeline de ingestão completo ──────────────────────────

# 1. EXTRACT — lê o PDF
raw_text = extract_text_from_pdf(pdf_path)
print(f"📄 Texto extraído: {len(raw_text)} caracteres")

# 2. TRANSFORM — chunking
chunks = chunk_text(raw_text, chunk_size=500, overlap=50)
print(f"✂️  Chunks gerados: {len(chunks)}")

# 3. TRANSFORM — embedding
texts = [c["text"] for c in chunks]
embeddings = generate_embeddings(texts)
print(f"🔢 Embeddings gerados: {len(embeddings)} vetores de {len(embeddings[0])} dimensões")

# 4. LOAD — upsert no Qdrant
points = []
for chunk, embedding in zip(chunks, embeddings):
    point_id = deterministic_id(chunk["text"])
    points.append(
        PointStruct(
            id=point_id,
            vector=embedding,
            payload={
                "text": chunk["text"],
                "chunk_index": chunk["chunk_index"],
                "source": pdf_path,
            },
        )
    )

qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"✅ {len(points)} pontos inseridos no Qdrant!")

# Verificando
info = qdrant.get_collection(COLLECTION)
print(f"📊 Total de pontos na collection: {info.points_count}")

---
## 🔍 Parte 6 — Retrieval: Buscando Chunks Relevantes

Agora que os chunks estão no Qdrant, vamos buscar os mais relevantes para uma pergunta.

O fluxo é:
1. Pergunta do usuário → embedding (mesma dimensão dos chunks)
2. Busca no Qdrant os K vetores mais próximos (nearest neighbors)
3. Retorna os chunks + score de similaridade

In [ ]:
# ── Testando a busca semântica ────────────────────────────

pergunta = "O que é um pipeline de ETL e quais ferramentas usar?"

# 1. Embedding da pergunta
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=[pergunta],
)
query_embedding = query_response.data[0].embedding

# 2. Busca no Qdrant (top 3 mais similares)
results = qdrant.search(
    collection_name=COLLECTION,
    query_vector=query_embedding,
    limit=3,
)

# 3. Resultados
print(f"🔍 Pergunta: {pergunta}\n")
for i, hit in enumerate(results, 1):
    print(f"--- Resultado {i} (score: {hit.score:.4f}) ---")
    print(f"{hit.payload['text'][:200]}...")
    print()

---
## 🤖 Parte 7 — RAG Completo: Retrieval + Augmented Generation

Agora juntamos retrieval + LLM. O "Augmented" do RAG significa que o prompt
é **aumentado** com o contexto recuperado do banco vetorial.

### Por que RAG ao invés de só perguntar pro LLM?

| Abordagem | Prós | Contras |
|-----------|------|--------|
| LLM puro | Simples | Alucina, sem fontes, desatualizado |
| RAG | Fundamentado, rastreável, atualizado | Mais complexo, depende da qualidade do retrieval |

RAG é o padrão de mercado para aplicações que precisam de **respostas confiáveis** baseadas em documentos.

In [ ]:
def query_rag(question: str, top_k: int = 5) -> dict:
    """
    Pipeline RAG completo:
      1. Embedding da pergunta
      2. Retrieval no Qdrant
      3. Monta prompt aumentado
      4. Gera resposta com LLM
    """
    # 1. Embedding
    q_emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question],
    ).data[0].embedding

    # 2. Retrieval
    hits = qdrant.search(
        collection_name=COLLECTION,
        query_vector=q_emb,
        limit=top_k,
    )

    # 3. Montar contexto
    context_parts = []
    for i, hit in enumerate(hits, 1):
        context_parts.append(
            f"[Trecho {i} | Score: {hit.score:.4f}]\n{hit.payload['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    augmented_prompt = f"""Com base nos trechos abaixo, responda à pergunta.

═══ CONTEXTO ═══
{context}

═══ PERGUNTA ═══
{question}

═══ INSTRUÇÃO ═══
Responda usando APENAS o contexto acima. Se não encontrar, diga."""

    # 4. Generation
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Você é um agente de estudos em Engenharia de Dados e IA. Responda em PT-BR."},
            {"role": "user", "content": augmented_prompt},
        ],
        temperature=0.3,
    )

    return {
        "answer": response.choices[0].message.content,
        "chunks_used": len(hits),
        "sources": list(set(h.payload.get("source", "?") for h in hits)),
    }


# ── Teste! ──────────────────────────────────────────────────
resultado = query_rag("Qual a diferença entre data lake e data warehouse?")

print("🤖 Resposta do Agente:")
print(resultado["answer"])
print(f"\n📚 Fontes: {resultado['sources']}")
print(f"📦 Chunks utilizados: {resultado['chunks_used']}")

In [ ]:
# 🧪 Teste com mais perguntas!

perguntas = [
    "O que é Apache Airflow e pra que serve?",
    "Como funciona streaming de dados?",
    "O que é RAG e como se conecta com engenharia de dados?",
]

for p in perguntas:
    print(f"\n{'='*60}")
    print(f"❓ {p}")
    print(f"{'='*60}")
    r = query_rag(p)
    print(r["answer"])
    print(f"\n📚 Fontes: {r['sources']}")

---
## 🚀 Parte 8 — FastAPI: Transformando em API REST

### O que é FastAPI?

FastAPI é um framework Python para construir APIs web. Ele é:
- **Rápido**: baseado em Starlette (async) e uvicorn (ASGI)
- **Tipado**: usa Pydantic para validação automática de dados
- **Documentado**: gera Swagger UI automaticamente em `/docs`

### Anatomia de um endpoint FastAPI

```python
@app.post("/query")              # 1. DECORATOR: define rota + método HTTP
async def query_documents(        # 2. FUNÇÃO: nome descritivo
    request: QueryRequest          # 3. PARÂMETRO: Pydantic valida automaticamente
) -> QueryResponse:               # 4. RETORNO: tipado = documentação automática
    result = query_rag(request.question)
    return result
```

### Métodos HTTP e quando usar

| Método | Uso | Exemplo |
|--------|-----|---------|
| GET | Ler/consultar dados | `/health`, `/stats` |
| POST | Criar/enviar dados | `/ingest`, `/query` |
| PUT | Atualizar dados | `/documents/{id}` |
| DELETE | Remover dados | `/documents/{id}` |

### Pydantic: Validação automática

```python
class QueryRequest(BaseModel):
    question: str = Field(
        ...,            # obrigatório
        min_length=3,   # mínimo 3 caracteres
        max_length=1000 # máximo 1000 caracteres
    )
```

Se alguém enviar `{"question": ""}`, FastAPI retorna automaticamente um erro 422 com detalhes.

### CORS: Permitindo o frontend acessar a API

Navegadores bloqueiam requests de um domínio para outro por segurança (Same-Origin Policy).
O CORS middleware diz ao navegador: "tá tudo bem, aceito requests desse frontend".

```python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # em produção, coloque o domínio do frontend
    allow_methods=["*"],
    allow_headers=["*"],
)
```

### O código completo está em `main.py`

Para rodar localmente:
```bash
uvicorn main:app --reload
```

Acesse `http://localhost:8000/docs` para o Swagger interativo.

---
## 🐳 Parte 9 — Deploy no Render

### Fluxo de deploy

```
Código local → git push → GitHub → Render detecta → Build Docker → Deploy → URL pública
```

### Passo a passo

1. **Crie um repo no GitHub** e faça push do projeto
2. **Acesse [render.com](https://render.com)** → New → Web Service
3. **Conecte o repositório GitHub**
4. **Configure:**
   - Runtime: Docker
   - Plan: Free
   - Health Check Path: `/health`
5. **Adicione variáveis de ambiente** no dashboard:
   - `OPENAI_API_KEY`
   - `QDRANT_URL`
   - `QDRANT_API_KEY`
6. **Deploy!** O Render builda a imagem Docker e expõe a URL

### O Dockerfile explicado

```dockerfile
FROM python:3.11-slim          # Imagem base leve do Python
WORKDIR /app                   # Diretório de trabalho no container
COPY requirements.txt .        # Copia só o requirements primeiro
RUN pip install -r ...         # Instala deps (cacheado se não mudar)
COPY . .                       # Copia o resto do código
EXPOSE 8000                    # Documenta a porta (informativo)
CMD ["uvicorn", ...]           # Comando de inicialização
```

**Por que copiar `requirements.txt` antes do código?**
Docker usa cache de layers. Se só o código mudou (e não as deps), o `pip install` não roda de novo. Isso acelera o build em 10x.

---
## 🎨 Parte 10 — Frontend com Lovable

O Lovable gera interfaces React a partir de prompts. O prompt para criar o frontend está no README do projeto.

O frontend vai:
1. Ter um campo de upload de PDF (chama `POST /ingest`)
2. Ter um campo de pergunta (chama `POST /query`)
3. Mostrar a resposta + fontes usadas

### Como conectar

No Lovable, o frontend faz `fetch()` para a URL do Render:
```javascript
const response = await fetch('https://sua-api.onrender.com/query', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ question: userQuestion }),
});
```

O CORS que configuramos no FastAPI permite essa comunicação cross-origin.

---
## 🏁 Recapitulação

Neste projeto você aprendeu:

| Conceito | O que fez | Ferramenta |
|----------|-----------|------------|
| Embeddings | Transformou texto em vetores numéricos | OpenAI API |
| Vector Store | Armazenou e buscou vetores por similaridade | Qdrant Cloud |
| Chunking | Fatiou documentos com janela deslizante | Python puro |
| RAG | Combinou retrieval + geração com LLM | OpenAI + Qdrant |
| API REST | Expôs tudo via endpoints HTTP | FastAPI |
| Containerização | Empacotou a app em Docker | Dockerfile |
| Deploy | Colocou em produção na nuvem | Render |
| Frontend | Criou interface para o usuário | Lovable |

### Próximos passos

- Adicionar autenticação (API key no header)
- Implementar hybrid search (BM25 + semântico)
- Adicionar re-ranking dos chunks (Cohere Rerank, por exemplo)
- Monitorar custo e latência por request
- Testar com documentos maiores e avaliar chunk_size ideal